# Yang: candidate score and DTW boundary experiments

This notebook is prepared for you to run; no dataset benchmark has been executed.
The default is the six earlier candidates, not the whole Yang release. Several
matches were weak; Yangjian is violin. The Erquan MIDI is an unverified piano
arrangement. See `../modules/vibrato/tests/README.md` for assumptions.

Run preparation, inspect the generated alignment images, then run the score arm.
The DTW diagnostic is disabled until you explicitly enable it.


In [1]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'app.py').is_file())
MODULE = 'benchmarks.modules.vibrato.tests.YangScore'
TAG = 'yang_score_boundary_v1'  # choose a new tag for a new experiment
PREPARED = ROOT / 'benchmarks/results' / f'{TAG}_inputs'
SCORE_RUN = ROOT / 'benchmarks/results' / f'{TAG}_score'
DTW_RUN = ROOT / 'benchmarks/results' / f'{TAG}_dtw'
MAPPING = ROOT / 'benchmarks/modules/vibrato/tests/yang_score_candidates.json'

def run(*args):
    subprocess.run([sys.executable, '-m', MODULE, *map(str, args)], cwd=ROOT, check=True)


In [2]:
run('prepare', '--mapping', MAPPING, '--output', PREPARED)


Preparing Huangjiangqin-1
Preparing Huangjiangqin-2
Preparing Huangjiangqin-3
Preparing Yangjian-1
Preparing Yangjian-2
Preparing Yangjian-3
Prepared 6 candidates; 70 unmapped recordings. No F1 scoring performed.


Inspect each `alignment.png` and `manifest.json` in PREPARED before interpreting
scores. `score.mid` preserves symbolic rhythm and repeats; `dtw_bounds.json`
contains separate, estimated audio-time boundaries. Neither is verified truth.


In [3]:
run('run', '--prepared', PREPARED, '--mode', 'score', '--output', SCORE_RUN, '--workers', 2)


common_baselines
 Method  Overall Soft F1  Extent Soft F1  Rate Soft F1  Extent Soft Precision  Extent Soft Recall  Rate Soft Precision  Rate Soft Recall  Matched Parameter Notes (Yang)  Annotated Parameter Notes (Yang)  Parameter Coverage (Yang)  Note Precision (Yang)  Note Recall (Yang)  Note F1 (Yang)  Frame Precision (recording mean)  Frame Recall (recording mean)  Frame F1 (recording mean)  Note Precision (recording mean)  Note Recall (recording mean)  Note F1 (recording mean)  Evaluated Recordings (Yang)  Parameter Recordings (Yang)  Reference Vibratos (Yang)  Detected Vibratos (Yang)  Only Bad Onset Rate (overlap)  Only Bad Offset Rate (overlap)  Split Rate (overlap)  Merge Rate (overlap)  Spurious Rate (overlap)  Non-Detected Rate (overlap)  Overall F1 (hard)  Overall Precision  Overall Recall  Extent F1 (hard)  Extent Precision  Extent Recall  Extent Accuracy (Yang)  Rate F1 (hard)  Rate Precision  Rate Recall  Rate Accuracy (Yang)  Center Accuracy (Attune)  Detection F1  Dete

In [4]:
columns = ['arm', 'method', 'frame_f1', 'yang_note_f1', 'aggregate_soft_f1']
display(pd.read_csv(SCORE_RUN / 'comparison.csv')[columns])


,arm,method,frame_f1,yang_note_f1,aggregate_soft_f1
0,common_baselines,attune,0.653835,0.089286,0.508415
1,common_baselines,yang_dt,0.842979,0.539474,0.669588
2,common_baselines,yang_br,0.595989,0.361233,0.472640
3,attune_score,attune,0.651041,0.093842,0.507134


If the score-assisted result is still below Yang on your prespecified metric,
set `RUN_DTW = True` below. This bypasses note detection and repeat recovery and
uses DTW boundaries with exactly the same pitch and truth. It is a segmentation
diagnostic, not a perfect-boundary oracle. Do not compare across different cohorts.


In [6]:
RUN_DTW = True
if RUN_DTW:
    run('run', '--prepared', PREPARED, '--mode', 'dtw', '--output', DTW_RUN, '--workers', 2)
    display(pd.concat([pd.read_csv(SCORE_RUN / 'comparison.csv'), pd.read_csv(DTW_RUN / 'comparison.csv')], ignore_index=True)[columns])


attune_dtw
Method  Overall Soft F1  Extent Soft F1  Rate Soft F1  Extent Soft Precision  Extent Soft Recall  Rate Soft Precision  Rate Soft Recall  Matched Parameter Notes (Yang)  Annotated Parameter Notes (Yang)  Parameter Coverage (Yang)  Note Precision (Yang)  Note Recall (Yang)  Note F1 (Yang)  Frame Precision (recording mean)  Frame Recall (recording mean)  Frame F1 (recording mean)  Note Precision (recording mean)  Note Recall (recording mean)  Note F1 (recording mean)  Evaluated Recordings (Yang)  Parameter Recordings (Yang)  Reference Vibratos (Yang)  Detected Vibratos (Yang)  Only Bad Onset Rate (overlap)  Only Bad Offset Rate (overlap)  Split Rate (overlap)  Merge Rate (overlap)  Spurious Rate (overlap)  Non-Detected Rate (overlap)  Overall F1 (hard)  Overall Precision  Overall Recall  Extent F1 (hard)  Extent Precision  Extent Recall  Extent Accuracy (Yang)  Rate F1 (hard)  Rate Precision  Rate Recall  Rate Accuracy (Yang)  Center Accuracy (Attune)  Detection F1  Detection P

,arm,method,frame_f1,yang_note_f1,aggregate_soft_f1
0,common_baselines,attune,0.653835,0.089286,0.508415
1,common_baselines,yang_dt,0.842979,0.539474,0.669588
2,common_baselines,yang_br,0.595989,0.361233,0.472640
3,attune_score,attune,0.651041,0.093842,0.507134
4,attune_dtw,attune,0.570263,0.036101,0.422078
